# Notebook 1: Setup & Preprocessing

**What this does (in simple terms):**

Think of this as preparing ingredients before cooking.
Raw audio files come in different lengths and formats.
This notebook makes them all the same — same length, same format —
so the model can process them in neat batches later.

Steps:
1. Connect Google Drive (to save checkpoints later)
2. Download the ASVspoof 2019 dataset to Colab's fast local disk
3. Analyze audio durations (to pick the right length)
4. Preprocess all audio: convert to mono, resample to 16kHz, pad/truncate to 6s
5. Save preprocessed files
6. Delete raw data to free up space

**Run this on Colab. Takes ~20-30 minutes total.**

## 1.1 Mount Google Drive & Install Libraries

In [ ]:
# Mount Google Drive — checkpoints and results will be saved here
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/deepfake_project/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/deepfake_project/results', exist_ok=True)
print("Google Drive mounted!")
print("Checkpoint dir: /content/drive/MyDrive/deepfake_project/checkpoints")

In [ ]:
# Install required libraries (run once per session)
!pip install librosa soundfile tqdm matplotlib pandas -q

## 1.2 Download ASVspoof 2019 LA Dataset

This downloads the dataset directly to Colab's local disk (~100GB available).
The download takes ~5-10 minutes depending on connection speed.

In [ ]:
import os
from pathlib import Path

RAW_DATA_DIR = Path("/content/raw_data")
PREPROCESSED_DIR = Path("/content/preprocessed")

RAW_DATA_DIR.mkdir(exist_ok=True)
PREPROCESSED_DIR.mkdir(exist_ok=True)

# Download dataset
# NOTE: If this URL doesn't work, you can:
# 1. Upload the zip file to your Google Drive
# 2. Copy it: !cp /content/drive/MyDrive/LA.zip /content/raw_data/LA.zip
print("Downloading ASVspoof 2019 LA dataset...")
print("This takes ~5-10 minutes. Be patient.\n")

!wget -q --show-progress -O /content/raw_data/LA.zip "https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip"

In [ ]:
# Extract the zip file
print("Extracting dataset...")
!cd /content/raw_data && unzip -q LA.zip
print("Done!")

# Check what's inside
!ls /content/raw_data/LA/

In [ ]:
# Set paths to the extracted data
INPUT_ROOT = RAW_DATA_DIR / "LA"

# Verify the folder structure
TRAIN_AUDIO = INPUT_ROOT / "ASVspoof2019_LA_train" / "flac"
DEV_AUDIO = INPUT_ROOT / "ASVspoof2019_LA_dev" / "flac"
EVAL_AUDIO = INPUT_ROOT / "ASVspoof2019_LA_eval" / "flac"
PROTOCOL_DIR = INPUT_ROOT / "ASVspoof2019_LA_cm_protocols"

for name, path in [("Train audio", TRAIN_AUDIO), ("Dev audio", DEV_AUDIO),
                    ("Eval audio", EVAL_AUDIO), ("Protocols", PROTOCOL_DIR)]:
    exists = path.exists()
    count = len(list(path.glob("*"))) if exists else 0
    print(f"{name}: {'FOUND' if exists else 'MISSING'} ({count} files)")

## 1.3 Duration Analysis

Before we decide how long to make each audio clip, let's look at
how long the actual audio files are. This helps us pick a smart
cutoff instead of guessing.

In [ ]:
import numpy as np
import librosa
from tqdm import tqdm
import matplotlib.pyplot as plt

TARGET_SR = 16000

AUDIO_FOLDERS = {
    "Train": INPUT_ROOT / "ASVspoof2019_LA_train" / "flac",
    "Dev": INPUT_ROOT / "ASVspoof2019_LA_dev" / "flac",
    "Eval": INPUT_ROOT / "ASVspoof2019_LA_eval" / "flac",
}

all_durations = []

for split_name, folder in AUDIO_FOLDERS.items():
    files = list(folder.glob("*.flac"))
    durations = []
    for f in tqdm(files, desc=f"Scanning {split_name}"):
        duration = librosa.get_duration(path=str(f))
        durations.append(duration)
    all_durations.extend(durations)

all_durations = np.array(all_durations)

In [ ]:
# Print statistics
print(f"Total files: {len(all_durations)}")
print(f"Shortest:    {all_durations.min():.2f}s")
print(f"Longest:     {all_durations.max():.2f}s")
print(f"Average:     {all_durations.mean():.2f}s")
print(f"Median:      {np.median(all_durations):.2f}s")
print()
for p in [90, 95, 98, 99]:
    val = np.percentile(all_durations, p)
    print(f"  {p}th percentile: {val:.2f}s")

In [ ]:
# Plot duration distribution
plt.figure(figsize=(14, 5))
plt.hist(all_durations, bins=80, color="steelblue", edgecolor="white", alpha=0.85)
plt.axvline(6.0, color="red", linestyle="--", linewidth=2, label="Our cutoff: 6.0s (95th pct)")
plt.axvline(all_durations.mean(), color="green", linestyle="--", linewidth=2,
            label=f"Mean: {all_durations.mean():.2f}s")
plt.xlabel("Duration (seconds)")
plt.ylabel("Number of Files")
plt.title("Audio Duration Distribution")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# We choose 6 seconds (95th percentile)
# This means only ~5% of files get truncated
MAX_DURATION = 6
MAX_SAMPLES = TARGET_SR * MAX_DURATION  # 96000

truncated = np.sum(all_durations > MAX_DURATION)
print(f"MAX_DURATION = {MAX_DURATION}s ({MAX_SAMPLES} samples)")
print(f"Files truncated: {truncated} ({100*truncated/len(all_durations):.1f}%)")
print(f"Files padded:    {len(all_durations) - truncated} ({100*(1-truncated/len(all_durations)):.1f}%)")

## 1.4 Preprocess All Audio Files

For each audio file:
- Convert to mono (single channel)
- Resample to 16kHz (what Wav2Vec2 expects)
- Pad short files with silence OR truncate long files to 6 seconds
- Save the preprocessed file
- Record the original length (needed for masks during training)

In [ ]:
import soundfile as sf
import pandas as pd

def preprocess_one_file(input_path, output_path, max_samples=MAX_SAMPLES, target_sr=TARGET_SR):
    """
    Preprocess a single audio file.
    Returns metadata dict or None if failed.
    """
    try:
        # Load audio
        audio, sr = librosa.load(str(input_path), sr=None, mono=True)

        # Resample to 16kHz if needed
        if sr != target_sr:
            audio = librosa.resample(y=audio, orig_sr=sr, target_sr=target_sr)

        original_length = len(audio)

        # Pad or truncate to max_samples
        if original_length < max_samples:
            audio = np.pad(audio, (0, max_samples - original_length), mode="constant")
        else:
            audio = audio[:max_samples]
            original_length = max_samples

        # Save preprocessed audio
        output_path.parent.mkdir(parents=True, exist_ok=True)
        sf.write(str(output_path), audio, target_sr)

        return {
            "audio_id": input_path.stem,
            "original_length": original_length,
            "was_padded": original_length < max_samples,
            "was_truncated": original_length > max_samples,
            "status": "success"
        }
    except Exception as e:
        return {
            "audio_id": input_path.stem,
            "status": "failed",
            "error": str(e)
        }

In [ ]:
# Process all splits
SPLIT_FOLDERS = {
    "train": ("ASVspoof2019_LA_train/flac", "train"),
    "dev": ("ASVspoof2019_LA_dev/flac", "dev"),
    "eval": ("ASVspoof2019_LA_eval/flac", "eval"),
}

all_metadata = []

for split_name, (folder, out_name) in SPLIT_FOLDERS.items():
    input_folder = INPUT_ROOT / folder
    output_folder = PREPROCESSED_DIR / out_name
    files = sorted(input_folder.glob("*.flac"))

    print(f"\nProcessing {split_name}: {len(files)} files")

    for f in tqdm(files, desc=split_name):
        output_path = output_folder / f"{f.stem}.flac"
        result = preprocess_one_file(f, output_path)
        result["split"] = split_name
        all_metadata.append(result)

# Save metadata
metadata_df = pd.DataFrame(all_metadata)
metadata_df.to_csv(PREPROCESSED_DIR / "metadata.csv", index=False)

In [ ]:
# Summary
success = metadata_df[metadata_df["status"] == "success"]
failed = metadata_df[metadata_df["status"] == "failed"]

print(f"\nPreprocessing Complete!")
print(f"  Successful: {len(success)}")
print(f"  Failed:     {len(failed)}")
print(f"  Padded:     {success['was_padded'].sum()}")
print(f"  Truncated:  {success['was_truncated'].sum()}")

# Verify file counts
for split_name, (_, out_name) in SPLIT_FOLDERS.items():
    count = len(list((PREPROCESSED_DIR / out_name).glob("*.flac")))
    print(f"  {split_name}: {count} preprocessed files saved")

## 1.5 Copy Protocols to Preprocessed Directory

The protocol files tell us which audio is real (bonafide) and which is fake (spoof).
We copy them alongside the preprocessed audio for easy access.

In [ ]:
import shutil

proto_dest = PREPROCESSED_DIR / "protocols"
proto_dest.mkdir(exist_ok=True)

for proto_file in PROTOCOL_DIR.glob("*.txt"):
    shutil.copy2(str(proto_file), str(proto_dest / proto_file.name))
    print(f"Copied: {proto_file.name}")

## 1.6 Delete Raw Data to Free Space

The raw data takes ~18GB. Now that we have preprocessed files,
we don't need it anymore. This frees up Colab disk space.

In [ ]:
print(f"Disk usage BEFORE cleanup:")
!df -h /content | tail -1

print(f"\nDeleting raw data...")
!rm -rf /content/raw_data

print(f"\nDisk usage AFTER cleanup:")
!df -h /content | tail -1

In [ ]:
# Verify preprocessed data is still there
print("Preprocessed data:")
!du -sh /content/preprocessed/*

print(f"\nTotal preprocessed:")
!du -sh /content/preprocessed

## 1.7 Save Config

Save configuration so other notebooks use the same settings.

In [ ]:
import json

config = {
    "TARGET_SR": TARGET_SR,
    "MAX_DURATION": MAX_DURATION,
    "MAX_SAMPLES": MAX_SAMPLES,
    "PREPROCESSED_DIR": str(PREPROCESSED_DIR),
    "total_files_processed": len(success),
}

# Save to preprocessed dir (local)
with open(PREPROCESSED_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

# Also save to Google Drive (persists between sessions)
with open("/content/drive/MyDrive/deepfake_project/config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Config saved!")
print(json.dumps(config, indent=2))
print("\n✅ Notebook 1 complete. Proceed to Notebook 2.")